# Al Islam library: download books as txt, then build a Chroma RAG

1. **Download:** every book in `alislam_books_by_category.json` (made by `alislam_book_catalog.py`) is saved as `<book name>.txt` in one folder, `alislam_books_txt`, through the same text API as *Rohani khazain books in txt.ipynb*. All books are downloaded.
2. **Embed:** every page becomes **one** embedding with `gemini-embedding-2`, stored in a Chroma DB, 100 pages per request. Ruhani Khazain, Maktubat-e-Ahmad, Malfuzat and Barahin-e-Ahmadiyya are **not** embedded.

Both steps can be stopped and rerun at any time: finished books are skipped.

Run the cells in order. The download takes about 1.5 hours for ~482,000 pages.
 Embedding time and cost depend on your Gemini API quota.

In [ ]:
import os
import re
import json
import time
from getpass import getpass
from pathlib import Path

import requests

# ---------------- CONFIG ---------------- #

CATALOG_PATH = Path("alislam_books_by_category.json")
DOWNLOAD_DIR = Path("alislam_books_txt")
DOWNLOAD_MANIFEST = DOWNLOAD_DIR / "_download_manifest.json"

CHROMA_DIR = "./chroma_db/alislam_library"
EMBED_MANIFEST = Path(CHROMA_DIR) / "_embedded_books.json"

TEXT_API = "https://new.alislam.org/api/books/text"
PAGES_PER_REQUEST = 100     # the API accepts a comma-separated page list (100 pages return in ~0.1 s)
SECONDS_BETWEEN_REQUESTS = 1.0   # the crawl-delay in new.alislam.org/library/robots.txt

EMBED_MODEL = "gemini-embedding-2"
EMBED_DIM = 3072            # 768 or 1536 give a much smaller DB with little quality loss
EMBED_BATCH = 100           # pages per embedding request
EMBED_PAUSE_SECONDS = 2     # pause after each request, as in messiahbooks.ipynb
DOC_PREFIX = "title: {title} | text: {text}"      # document format for gemini-embedding-2
QUERY_PREFIX = "task: search result | query: {text}"

# Categories (exact names in the catalogue) that are downloaded but NOT embedded.
EXCLUDED_CATEGORIES = {
    "Ruhani Khazain",
    "Ruhani Khazain (English)",
    "Maktubat-e-Ahmad",
    "Malfuzat",
    "Malfuzat (English)",
    "Barahin-e-Ahmadiyya",
}
# Single books that are downloaded but not embedded, by book id.
# "ur-13017" is a separate Urdu edition of Lujjatun-Nur (Ruhani Khazain vol. 16) that is
# not in the Ruhani Khazain collection; remove it from this set to include it.
EXCLUDED_BOOK_IDS = {"ur-13017"}

# The key is read from the environment; otherwise you are asked for it (it is not saved).
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API key: ")

## 1. Choose the books

In [ ]:
catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))


def safe_name(text, limit=120):
    text = re.sub(r'[\\/:*?"<>|\n\r\t]+', " ", text)
    text = re.sub(r"\s+", " ", text).strip().strip(".")
    return (text[:limit].rstrip() or "untitled")


books = []        # every book with a page range: all are downloaded
for category, items in catalog.items():
    for book_id, info in items.items():
        if info["start_page"] is None or info["end_page"] is None:
            print("No page range, skipped:", book_id, info["name"])
            continue
        books.append(dict(info, book_id=book_id, category=category))

# Each book is saved as "<book name>.txt". A few names occur twice (e.g. an Urdu
# and an Arabic edition), so those files also get the book id.
name_counts = {}
for b in books:
    key = safe_name(b["name"]).lower()
    name_counts[key] = name_counts.get(key, 0) + 1
for b in books:
    stem = safe_name(b["name"])
    b["file_stem"] = stem if name_counts[stem.lower()] == 1 else f"{stem} [{safe_name(b['book_id'])}]"

embed_books = [b for b in books
               if b["category"] not in EXCLUDED_CATEGORIES and b["book_id"] not in EXCLUDED_BOOK_IDS]
not_embedded = [b for b in books if b not in embed_books]

total_pages = sum(b["end_page"] - b["start_page"] + 1 for b in books)
print(f"Books to download: {len(books)} ({total_pages:,} pages)")
print(f"Books to embed: {len(embed_books)}; downloaded but not embedded: {len(not_embedded)}")
for name in sorted({b["category"] for b in not_embedded}):
    print(f"  - {name}: {sum(b['category'] == name for b in not_embedded)} books")
print(f"Estimated download time: {total_pages / PAGES_PER_REQUEST * (SECONDS_BETWEEN_REQUESTS + 0.3) / 3600:.1f} hours")

## 2. Download every book as txt

Each file starts with the book's details, then one block per page:

```
--- Page 367 | print 337 ---
<page text>
```

`367` is the page number used by the API (and the catalogue's start/end page); `print 337` is the number printed on the page. Pages with no text are left out. Progress is in `alislam_books_txt/_download_manifest.json`; a book that had a failed request is not marked done, so rerunning this cell retries it.

In [ ]:
session = requests.Session()
session.headers["User-Agent"] = "Mozilla/5.0 (compatible; personal-library-download)"
_last_request = [0.0]


def book_path(book):
    return DOWNLOAD_DIR / f"{book['file_stem']}.txt"



def fetch_pages(book_id, page_numbers):
    """Return the API's list of page dicts, or None after repeated failures."""
    url = f"{TEXT_API}?id={requests.utils.quote(book_id)}&pages={','.join(map(str, page_numbers))}"
    for attempt in range(5):
        wait = SECONDS_BETWEEN_REQUESTS - (time.monotonic() - _last_request[0])
        if wait > 0:
            time.sleep(wait)
        _last_request[0] = time.monotonic()
        try:
            response = session.get(url, timeout=120)
        except requests.RequestException as exc:
            print(f"    network error ({type(exc).__name__}), retrying")
            time.sleep(10 * (attempt + 1))
            continue
        if response.status_code == 429 or response.status_code >= 500:
            print(f"    HTTP {response.status_code}, retrying")
            time.sleep(30 * (attempt + 1))
            continue
        if response.status_code != 200:
            print(f"    HTTP {response.status_code} for {book_id}")
            return None
        try:
            data = response.json()
        except ValueError:
            time.sleep(10)
            continue
        if isinstance(data, list):
            return data
        return None
    return None


def download_book(book):
    """Write the book's txt file. Returns (status, pages_with_text)."""
    numbers = list(range(book["start_page"], book["end_page"] + 1))
    pages, failed = {}, 0
    for i in range(0, len(numbers), PAGES_PER_REQUEST):
        chunk = numbers[i:i + PAGES_PER_REQUEST]
        data = fetch_pages(book["book_id"], chunk)
        if data is None:
            failed += 1
            continue
        for entry in data:
            text = (entry.get("content") or "").strip()
            if text:
                pages[int(entry.get("pageNum"))] = (str(entry.get("printPageNum") or ""), text)
    if failed:
        return "failed", len(pages)

    header = [
        f"Book: {book['name']}",
        f"Book (Urdu): {book.get('name_ur', '')}",
        f"Book ID: {book['book_id']}",
        f"Category: {book['category']}",
        f"Language: {book.get('language', '')}",
        f"Pages: {book['start_page']}-{book['end_page']}",
        f"Source: {book.get('url', '')}",
        "",
    ]
    body = []
    for number in sorted(pages):
        print_page, text = pages[number]
        body.append(f"--- Page {number} | print {print_page} ---\n{text}\n")
    path = book_path(book)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".txt.part")
    tmp.write_text("\n".join(header + body), encoding="utf-8")
    os.replace(tmp, path)
    return ("saved" if pages else "no_text"), len(pages)


DOWNLOAD_DIR.mkdir(exist_ok=True)
manifest = json.loads(DOWNLOAD_MANIFEST.read_text(encoding="utf-8")) if DOWNLOAD_MANIFEST.exists() else {}
todo = [b for b in books if manifest.get(b["book_id"], {}).get("status") not in ("saved", "no_text")]
print(f"{len(books) - len(todo)} books already downloaded, {len(todo)} to go")

for n, book in enumerate(todo, 1):
    started = time.time()
    status, count = download_book(book)
    manifest[book["book_id"]] = {"status": status, "pages_with_text": count, "file": str(book_path(book)),
                                 "category": book["category"], "name": book["name"]}
    DOWNLOAD_MANIFEST.write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"[{n}/{len(todo)}] {status:8} {count:5} pages  {book['name']}  ({time.time() - started:.0f}s)")

summary = {}
for entry in manifest.values():
    summary[entry["status"]] = summary.get(entry["status"], 0) + 1
print("Download totals:", summary)

## 3. Embedding with `gemini-embedding-2`

Only `embed_books` are embedded: the categories in `EXCLUDED_CATEGORIES` and the ids in `EXCLUDED_BOOK_IDS` are downloaded but skipped here.

Not the LangChain `GoogleGenerativeAIEmbeddings` class from *Chroma RAG.ipynb*: when `gemini-embedding-2` gets a plain list of texts it returns **one combined** embedding. Wrapping each page in its own `Content` returns one embedding per page.

`gemini-embedding-2` has no `task_type`. Pages are embedded as `title: <book> | text: <page>` and queries as `task: search result | query: <question>`, the formats in Google's embedding guide. Its vectors can't be compared with `gemini-embedding-001` ones, so this DB is separate from your other `chroma_db` folders.

In [ ]:
from google import genai
from google.genai import errors, types
from langchain_core.embeddings import Embeddings

genai_client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])


def _embed_once(texts):
    result = genai_client.models.embed_content(
        model=EMBED_MODEL,
        contents=[types.Content(parts=[types.Part.from_text(text=t)]) for t in texts],
        config=types.EmbedContentConfig(output_dimensionality=EMBED_DIM),
    )
    vectors = [e.values for e in result.embeddings]
    if len(vectors) != len(texts):
        raise RuntimeError(f"Asked for {len(texts)} embeddings, got {len(vectors)}")
    return vectors


def embed_texts(texts):
    """One embedding per text. Retries rate limits; splits a batch the API rejects.

    Returns a list with None for a single text that the API refuses (e.g. too long).
    """
    for attempt in range(8):
        try:
            return _embed_once(texts)
        except errors.APIError as exc:
            code = getattr(exc, "code", None)
            if code in (401, 402, 403):
                # Bad key, no credits or no permission: stop rather than skip every page.
                raise RuntimeError(f"Gemini refused the request ({code}). Fix the key or billing, then rerun. "
                                   f"Details: {str(exc)[:300]}") from exc
            if code == 429 or (code or 0) >= 500:

                wait = min(300, 15 * 2 ** attempt)
                print(f"    Gemini {code}, waiting {wait}s")
                time.sleep(wait)
                continue
            if len(texts) > 1:
                middle = len(texts) // 2
                return embed_texts(texts[:middle]) + embed_texts(texts[middle:])
            print(f"    page refused by Gemini ({code}): {str(exc)[:150]}")
            return [None]
    raise RuntimeError("Gemini kept rate-limiting; rerun this cell later to continue.")


class GeminiEmbedding2(Embeddings):
    """LangChain wrapper, so Chroma retrievers embed queries the same way."""

    def embed_documents(self, texts):
        return embed_texts([DOC_PREFIX.format(title="none", text=t) for t in texts])

    def embed_query(self, text):
        return embed_texts([QUERY_PREFIX.format(text=text)])[0]


gemini2_embeddings = GeminiEmbedding2()
print("Test embedding length:", len(gemini2_embeddings.embed_query("test")))

## 4. Build the Chroma DB: one embedding per page

Each page is stored with metadata `book`, `book_id`, `category`, `language`, `page` (the printed page number), `page_index` (the API page number) and `source`. IDs are `<book_id>:<page_index>`, so pages already in the DB are skipped on a rerun.

In [ ]:
from langchain_community.vectorstores import Chroma

PAGE_MARKER = re.compile(r"^--- Page (\d+) \| print (.*?) ---$", re.M)


def read_book_pages(path):
    """Yield (page_index, print_page, text) from a downloaded txt file."""
    text = Path(path).read_text(encoding="utf-8")
    parts = PAGE_MARKER.split(text)
    for i in range(1, len(parts), 3):
        content = parts[i + 2].strip()
        if content:
            yield int(parts[i]), parts[i + 1].strip(), content


vectorstore = Chroma(
    collection_name="langchain",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini2_embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
collection = vectorstore._collection

manifest = json.loads(DOWNLOAD_MANIFEST.read_text(encoding="utf-8")) if DOWNLOAD_MANIFEST.exists() else {}
embedded = json.loads(EMBED_MANIFEST.read_text(encoding="utf-8")) if EMBED_MANIFEST.exists() else {}
ready = [b for b in embed_books if manifest.get(b["book_id"], {}).get("status") == "saved"]
todo = [b for b in ready if b["book_id"] not in embedded]
print(f"{len(ready)} downloaded books with text; {len(embedded)} already embedded; {len(todo)} to go")

for n, book in enumerate(todo, 1):
    pages = list(read_book_pages(manifest[book["book_id"]]["file"]))
    ids = [f"{book['book_id']}:{index}" for index, _, _ in pages]
    have = set(collection.get(ids=ids, include=[])["ids"]) if ids else set()
    pending = [(i, p) for i, p in zip(ids, pages) if i not in have]
    refused = 0
    for start in range(0, len(pending), EMBED_BATCH):
        batch = pending[start:start + EMBED_BATCH]
        vectors = embed_texts([DOC_PREFIX.format(title=book["name"], text=p[2]) for _, p in batch])
        time.sleep(EMBED_PAUSE_SECONDS)

        keep = [(i, p, v) for (i, p), v in zip(batch, vectors) if v is not None]
        refused += len(batch) - len(keep)
        if keep:
            collection.add(
                ids=[i for i, _, _ in keep],
                embeddings=[v for _, _, v in keep],
                documents=[p[2] for _, p, _ in keep],
                metadatas=[{
                    "book": book["name"],
                    "book_id": book["book_id"],
                    "category": book["category"],
                    "language": book.get("language", ""),
                    "page": p[1] or str(p[0]),
                    "page_index": p[0],
                    "source": book.get("url", ""),
                } for _, p, _ in keep],
            )
    embedded[book["book_id"]] = {"pages": len(pages), "refused": refused}
    EMBED_MANIFEST.write_text(json.dumps(embedded, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"[{n}/{len(todo)}] {len(pages):5} pages ({len(have)} already in DB, {refused} refused)  {book['name']}")

print("Pages in the DB:", collection.count())

## 5. Search

In [ ]:
vectorstore_disk = Chroma(
    collection_name="langchain",
    persist_directory=CHROMA_DIR,
    embedding_function=gemini2_embeddings,
    collection_metadata={"hnsw:space": "cosine"},
)
retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 20})

# Optional: search inside one category, e.g.
# retriever = vectorstore_disk.as_retriever(search_kwargs={"k": 20, "filter": {"category": "Khutbat-e-Mahmood"}})

docs = retriever.invoke("""
Polygamy
""")
for d in docs:
    m = d.metadata
    print(f"{m['book']} | p. {m['page']} | {m['category']}")
    print(d.page_content)
    print("-" * 40)

## 6. Word / sentence search

Lexical search, as in `Chroma RAG.ipynb`: finds pages whose stored text contains the exact words,
instead of pages with a similar meaning. It needs no API calls.

- `mode="sentence"`: the exact phrase must appear on the page.
- `mode="word"`: every space-separated word must appear on the page, in any order.

Matching is exact and case-sensitive, and Urdu/Arabic diacritics (زیر، زبر) must match too, so try the
phrase with and without them. `category=` or `book_ids=` limits the search, which is also faster.

In [ ]:
# === Word / sentence matching in the Al Islam library Chroma DB ===
# Complements the semantic search above: finds exact substring matches in stored page text.

import html
import re

from IPython.display import HTML, display

lexical_collection = vectorstore_disk._collection
_ARABIC_SCRIPT = re.compile(r"[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]")
_LATIN = re.compile(r"[A-Za-z]")


def highlight_text(text, query, mode="sentence"):
    """Wrap matching phrase(s) in yellow <mark> tags for notebook display."""
    if not query:
        return html.escape(text)

    terms = [query] if mode == "sentence" else [t for t in query.split() if t]

    spans = []
    for term in terms:
        start = 0
        while True:
            idx = text.find(term, start)
            if idx == -1:
                break
            spans.append((idx, idx + len(term)))
            start = idx + 1

    if not spans:
        return html.escape(text)

    spans.sort()
    merged = [spans[0]]
    for start, end in spans[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))

    parts = []
    pos = 0
    for start, end in merged:
        if pos < start:
            parts.append(html.escape(text[pos:start]))
        parts.append(
            f'<mark style="background:#ffeb3b;padding:0 2px">{html.escape(text[start:end])}</mark>'
        )
        pos = end
    if pos < len(text):
        parts.append(html.escape(text[pos:]))

    return "".join(parts)


def match_in_library(query, mode="sentence", limit=20, category=None, book_ids=None):
    """
    Search the library DB for lexical matches.

    mode='sentence' -> exact phrase must appear in page text
    mode='word'     -> every whitespace-separated token must appear (any order)
    category        -> optional category name, e.g. "Khutbat-e-Mahmood"
    book_ids        -> optional list of book ids, e.g. ["5459-13245"]
    """
    query = query.strip()
    if not query:
        return []

    if mode == "sentence":
        where_document = {"$contains": query}
    elif mode == "word":
        tokens = query.split()
        where_document = (
            {"$contains": tokens[0]}
            if len(tokens) == 1
            else {"$and": [{"$contains": token} for token in tokens]}
        )
    else:
        raise ValueError("mode must be 'sentence' or 'word'")

    filters = []
    if category:
        filters.append({"category": category})
    if book_ids:
        filters.append({"book_id": {"$in": list(book_ids)}})
    where = None if not filters else filters[0] if len(filters) == 1 else {"$and": filters}

    result = lexical_collection.get(
        where=where,
        where_document=where_document,
        limit=limit,
        include=["documents", "metadatas"],
    )

    matches = []
    for doc, meta in zip(result["documents"], result["metadatas"]):
        matches.append(
            {
                "book": meta.get("book"),
                "book_id": meta.get("book_id"),
                "category": meta.get("category"),
                "page": meta.get("page"),
                "page_index": meta.get("page_index"),
                "text": doc,
                "query": query,
                "mode": mode,
            }
        )
    # Group the hits by book, in page order.
    matches.sort(key=lambda m: (m["book"] or "", int(m["page_index"] or 0)))
    return matches


def print_matches(matches, query=None, mode="sentence"):
    """Print each match with the complete page text and highlighted query."""
    if not matches:
        print("No matches.")
        return

    print(f"\n{'=' * 60}\nTotal matches: {len(matches)}\n{'=' * 60}")
    for i, match in enumerate(matches, 1):
        ref = f"[{match['book']} — p. {match['page']} — {match['category']}]"

        match_query = query or match.get("query", "")
        match_mode = match.get("mode", mode)
        highlighted = highlight_text(match["text"], match_query, mode=match_mode)
        rtl = len(_ARABIC_SCRIPT.findall(match["text"])) > len(_LATIN.findall(match["text"]))
        direction = "direction:rtl;text-align:right" if rtl else "direction:ltr;text-align:left"

        print(f"\n--- {i}. {ref} ---")
        display(
            HTML(
                f'<div style="{direction};white-space:pre-wrap;'
                f'font-family:inherit;line-height:1.6">{highlighted}</div>'
            )
        )
        print("-" * 40)


print(f"Pages available for word/sentence search: {lexical_collection.count()}")

In [ ]:
# --- Example: sentence matching across the whole library ---
QUERY = "وفات مسیح"
MODE = "sentence"  # "sentence" for exact phrase, "word" for all tokens

matches = match_in_library(QUERY, mode=MODE, limit=200)
print_matches(matches, query=QUERY, mode=MODE)

In [ ]:
# --- Example: word matching in one category ---
QUERY = "خلافت دعا"
CATEGORY = "Khutbat-e-Masroor"
MODE = "word"

matches = match_in_library(QUERY, mode=MODE, limit=20, category=CATEGORY)
print_matches(matches, query=QUERY, mode=MODE)